# RLVR: Kaggle T4 sessions (FP16 numerical variant)

The full scientific target stays at 120 outer steps. Each invocation runs a bounded slice, saving a verified session snapshot. `PAUSED` means resume is required, not a completed result. No persistent-failure or new-method result has been established.

Select **GPU T4 x2**, enable Internet, and enable **Files only persistence before starting**. This implementation uses GPU 0 only; two 16 GB GPUs do not form one 32 GB GPU. Environments, downloads and the active run stay under `/tmp`; only the snapshot vault occupies `/kaggle/working`. Save a notebook version with outputs after a slice. For a new version/session, attach the previous notebook's outputs and set `PRIOR_VAULT` below.

The default FP16 path uses FP32 master weights/AdamW moments and dynamic loss scaling. It changes the BF16 numerical recipe and must be reported separately. It has CPU correctness tests; a real T4 run is still required to establish GPU feasibility and throughput.


## 1. Configure

If the repository is private, add a read-only `GITHUB_TOKEN` using the platform secret UI, then enable `USE_GITHUB_SECRET`. Tokens are passed to Git through an ephemeral askpass helper, never embedded in notebook URLs. Keep the same numerical settings when resuming. Changing seeds/precision uses separate vaults.

In [ ]:
from pathlib import Path
PLATFORM="kaggle"
PRECISION="float16"  # bfloat16 on L4/A100; float16 is a separate T4 variant
SEED=1
SESSION_HOURS=6.0  # Training budget only; installation and uploads consume runtime too
PAUSE_MARGIN_MINUTES=120
CONTINUE_EXPORTED_STEP1=False  # False starts from the pinned base model
REF="main"
USE_GITHUB_SECRET=False  # True for a private repository; create a GITHUB_TOKEN platform secret
CODE_OVERLAY=""  # Optional notebook_support.zip upload/input if GitHub lacks these changes
ARTIFACT_DIR=""  # Optional attached dataset containing manifest.json and all six LFS parts
PRIOR_VAULT=""  # Kaggle: /kaggle/input/<previous-notebook-output>/vault_float16_seed1
WORK_ROOT=Path("/tmp/rlvr_runtime")
PROJECT=Path("/tmp/rlvr_success_transfer")
RUN=WORK_ROOT/f"run_{PRECISION}_seed{SEED}"


In [ ]:
VAULT=Path("/kaggle/working")/f"vault_{PRECISION}_seed{SEED}"


## 3. Get the code

This clones ordinary files with LFS smudging disabled. If the notebook update is not yet on GitHub, Colab prompts for the supplied code overlay ZIP; on Kaggle attach that ZIP as input and set its path in `CODE_OVERLAY`.

In [ ]:
import os, sys, subprocess, tempfile, zipfile, shutil
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['HF_HOME']=str(WORK_ROOT/'huggingface')
os.environ['HF_HUB_DISABLE_XET']='1'
WORK_ROOT.mkdir(parents=True,exist_ok=True)
TOKEN=None
if USE_GITHUB_SECRET:
    if PLATFORM=='colab':
        from google.colab import userdata
        TOKEN=userdata.get('GITHUB_TOKEN')
    else:
        from kaggle_secrets import UserSecretsClient
        TOKEN=UserSecretsClient().get_secret('GITHUB_TOKEN')
if not PROJECT.exists():
    with tempfile.TemporaryDirectory(dir=WORK_ROOT) as tmp:
        env=os.environ.copy();env.update(GIT_LFS_SKIP_SMUDGE='1',GIT_TERMINAL_PROMPT='0')
        if TOKEN:
            askpass=Path(tmp)/'askpass.py'
            askpass.write_text('#!'+sys.executable+'\nimport os,sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["RLVR_GITHUB_TOKEN"])\n')
            askpass.chmod(0o700);env.update(GIT_ASKPASS=str(askpass),RLVR_GITHUB_TOKEN=TOKEN)
        subprocess.run(['git','-c','filter.lfs.process=','-c','filter.lfs.smudge=',
            '-c','filter.lfs.required=false','clone','--depth','1','--branch',REF,
            'https://github.com/tuananhlevan/rlvr_success_transfer.git',str(PROJECT)],check=True,env=env)
# Offline delivery fallback: useful if the notebook changes have not reached GitHub.
if CODE_OVERLAY or not (PROJECT/'scripts/notebook_session.py').exists():
    if not CODE_OVERLAY and PLATFORM=='colab':
        print('Upload notebook_support.zip supplied with this notebook.')
        from google.colab import files
        uploaded=files.upload()
        choices=[name for name in uploaded if name.endswith('.zip')]
        if len(choices)!=1:raise RuntimeError('Upload one notebook_support.zip file')
        CODE_OVERLAY=choices[0]
    if not CODE_OVERLAY:raise RuntimeError('Attach notebook_support.zip as Kaggle input and set CODE_OVERLAY to its path.')
    with zipfile.ZipFile(CODE_OVERLAY) as archive:
        if sum(item.file_size for item in archive.infolist())>20*1024**2:raise ValueError('Overlay is unexpectedly large')
        for item in archive.infolist():
            target=(PROJECT/item.filename).resolve()
            if not target.is_relative_to(PROJECT.resolve()) or (item.external_attr>>16)&0o170000==0o120000:
                raise ValueError('Unsafe overlay path')
            if item.is_dir():target.mkdir(parents=True,exist_ok=True);continue
            target.parent.mkdir(parents=True,exist_ok=True)
            with archive.open(item) as source,target.open('wb') as output:shutil.copyfileobj(source,output)
print('Code ready at',PROJECT)


## 4. Install and check isolated environments

No replacement of the notebook kernel packages is needed. This downloads managed Python 3.11 plus separate training/inference environments. It can take substantial time and disk. The final doctor executes GPU kernels; stop if it fails. T4 uses xFormers, not FlashInfer.

In [ ]:
command=[sys.executable,str(PROJECT/'scripts/bootstrap_notebook.py'),
    '--work-root',str(WORK_ROOT),'--precision',PRECISION]
if PRECISION=='bfloat16':command.append('--prebuilt-kernels')
subprocess.run(command,check=True)


## 5. Restore state or prepare the migration

Fresh Kaggle FP16 seeds are the default to avoid mixing a BF16 checkpoint into the FP16 experiment. Enabling the step-1 migration costs about 5.5 GB of LFS download bandwidth and records a precision transition if used on T4. Subsequent sessions restore the saved vault without downloading it again.

In [ ]:
sys.path.insert(0,str(PROJECT/'scripts'))
from session_store import candidates, restore
if PRIOR_VAULT and not candidates(VAULT) and not (RUN/'inputs.json').exists():
    restore(Path(PRIOR_VAULT),RUN)
# Download the large migration only once, if explicitly requested and no saved state exists.
ARTIFACT_PATH=Path(ARTIFACT_DIR) if ARTIFACT_DIR else PROJECT/'artifacts'
if CONTINUE_EXPORTED_STEP1 and not candidates(VAULT) and not (RUN/'inputs.json').exists():
    if not ARTIFACT_DIR:
        env=os.environ.copy()
        if TOKEN:env['RLVR_GITHUB_TOKEN']=TOKEN
        subprocess.run([sys.executable,str(PROJECT/'scripts/artifacts.py'),'download'],check=True,env=env)
    print('Migration selected: the launcher will verify and restore the exported step-1 state.')
else:
    print('Using an existing saved session, or starting a fresh seed; no migration download.')


## 6. Run one session slice

A 120-minute margin stops new phases near the budget. Generation also checkpoints chunks. A phase that cannot finish before the hard deadline may be terminated; uncommitted optimizer work replays from the saved buffer. Session termination can happen earlier than your selected budget. Do not change `--steps` to fit a session: the slice budget controls time while the scientific horizon remains 120.

In [ ]:
command=[sys.executable,str(PROJECT/'scripts/notebook_session.py'),
    '--run-dir',str(RUN),'--vault',str(VAULT),'--precision',PRECISION,
    '--hours',str(SESSION_HOURS),'--pause-margin-minutes',str(PAUSE_MARGIN_MINUTES),
    '--seed',str(SEED),'--artifact-dir',str(ARTIFACT_PATH)]
if CONTINUE_EXPORTED_STEP1:command.append('--migration')
subprocess.run(command,check=True)


## 7. Inspect the result and save outputs

On Kaggle, save outputs and attach them next time, or resume the same persisted editing session. A new `Save & Run All` starts clean, so attach the prior output explicitly. On Colab, rerun this notebook on a fresh runtime using the same vault. Do not place two concurrent writers on one vault.

In [ ]:
import json
status=RUN/'status.json'
if status.exists():print(status.read_text())
for path in sorted(RUN.glob('seed_*/latest_checkpoint/commit.json')):
    print(path.parent.parent.name,json.loads(path.read_text()))
log=RUN/'worker.log'
if log.exists():print('\n'.join(log.read_text(errors='replace').splitlines()[-25:]))
print('Persistent snapshot:',VAULT)


## Limits and sources

The full three-seed reproduction needs many sessions; quotas and GPU availability can change. This notebook does not bypass service session limits or quotas. The 5060 Ti timing does not predict T4 runtime. A GPU doctor/benchmark is an engineering check, not a learning result.

Sources: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Kaggle notebook documentation](https://www.kaggle.com/docs/notebooks), [Kaggle T4 hardware notice](https://www.kaggle.com/product-announcements/735239), [vLLM 0.11.2 CUDA backends](https://docs.vllm.ai/en/v0.11.2/api/vllm/platforms/cuda/). See `CLOUD_RUN.md` for backup semantics and verification scope.